# 🕷️ Spider Intelligence — Brain V0.1

A tiny character-level Transformer initialized from random weights and trained from scratch. This is an educational first experiment—not a capable assistant yet.

Run the cells from top to bottom. In Colab, optionally choose **Runtime → Change runtime type → GPU**.

In [ ]:
import math, torch
from torch import nn
import torch.nn.functional as F
from pathlib import Path

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 1. Training text

Replace this starter text with your own dataset later. More diverse, clean text is needed for meaningful language learning.

In [ ]:
text = '''Spider Intelligence is a project to build an AI system from the ground up.
A neural network learns patterns by adjusting parameters during training.
A tokenizer maps text characters to numbers and numbers back to characters.
Spider Brain predicts the next character from a sequence of previous characters.
Training data should be diverse, accurate, and legally obtained.
This is the beginning of Spider Intelligence. The model starts with random weights.
Learning requires examples, computation, experiments, and evaluation.
'''

In [ ]:
# Character tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda ids: ''.join(itos[int(i)] for i in ids)
data = torch.tensor(encode(text), dtype=torch.long)
print('Characters:', vocab_size, '| Tokens:', len(data))

## 2. Batch sampler
The model learns to predict the next character.

In [ ]:
block_size = 64
batch_size = 32
def get_batch():
    ix = torch.randint(len(data)-block_size-1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## 3. Transformer architecture
This is our own small architecture using PyTorch's tensor and attention primitives. All weights are randomly initialized; no pretrained model is loaded.

In [ ]:
class CausalBlock(nn.Module):
    def __init__(self, n_embd, n_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(n_embd, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(n_embd)
        self.norm2 = nn.LayerNorm(n_embd)
        self.ff = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(), nn.Linear(4*n_embd, n_embd))
    def forward(self, x):
        length = x.size(1)
        mask = torch.triu(torch.ones(length, length, device=x.device, dtype=torch.bool), diagonal=1)
        a,_ = self.attn(x,x,x,attn_mask=mask,need_weights=False)
        x = self.norm1(x+a)
        return self.norm2(x+self.ff(x))

class SpiderBrain(nn.Module):
    def __init__(self, vocab_size, block_size=64, n_embd=64, n_heads=4, n_layers=2):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size,n_embd)
        self.pos_emb = nn.Embedding(block_size,n_embd)
        self.blocks = nn.Sequential(*[CausalBlock(n_embd,n_heads) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd,vocab_size)
    def forward(self, idx, targets=None):
        B,T = idx.shape
        positions = torch.arange(T,device=idx.device)
        x = self.token_emb(idx)+self.pos_emb(positions)
        x = self.blocks(x)
        logits = self.head(self.final_norm(x))
        loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)),targets.reshape(-1)) if targets is not None else None
        return logits,loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:,-self.block_size:]
            logits,_ = self(idx_cond)
            logits = logits[:,-1,:]/max(temperature,1e-4)
            probs = F.softmax(logits,dim=-1)
            next_id = torch.multinomial(probs,1)
            idx = torch.cat((idx,next_id),dim=1)
        return idx


In [ ]:
model = SpiderBrain(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print(f'Parameters: {sum(p.numel() for p in model.parameters():,)}')

## 4. Train
The starter corpus is intentionally tiny. Training it demonstrates the pipeline only; it will mostly memorize the text.

In [ ]:
steps = 500
model.train()
for step in range(steps):
    xb,yb = get_batch()
    _,loss = model(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    optimizer.step()
    if step % 50 == 0:
        print(f'Step {step:4d}/{steps} | loss {loss.item():.4f}')
print('Training finished')

## 5. Generate text

In [ ]:
prompt = 'Spider'
prompt_ids = torch.tensor([encode(prompt)],dtype=torch.long,device=device)
output = model.generate(prompt_ids,max_new_tokens=250,temperature=0.8)
print(decode(output[0].tolist()))

## 6. Save checkpoint
Checkpoint includes model weights and tokenizer vocabulary.

In [ ]:
checkpoint = {'model_state':model.state_dict(),'chars':chars,'config':{'block_size':block_size,'n_embd':64,'n_heads':4,'n_layers':2}}
torch.save(checkpoint,'spider_brain_v0_1.pt')
print('Saved spider_brain_v0_1.pt')
from google.colab import files
files.download('spider_brain_v0_1.pt')